In [1]:
import math
from datetime import datetime

def solar_altitude_azimuth(year, month, day, hour, minute, latitude, longitude, timezone_offset):
    """
    Calculate solar altitude and azimuth angles.
    :param year: int
    :param month: int
    :param day: int
    :param hour: int (local clock time)
    :param minute: int
    :param latitude: float in degrees
    :param longitude: float in degrees
    :param timezone_offset: float (e.g. +2 for CEST)
    :return: (altitude_deg, azimuth_deg)
    """
    lat_rad = math.radians(latitude)

    # Day of the year
    date = datetime(year, month, day)
    n = date.timetuple().tm_yday

    # Fractional local time
    decimal_hour = hour + minute / 60.0

    # Declination δ
    decl = 23.45 * math.sin(math.radians(360/365 * (284 + n)))
    decl_rad = math.radians(decl)

    # Equation of time (in minutes)
    B = math.radians(360/365 * (n - 81))
    EoT = 9.87 * math.sin(2*B) - 7.53 * math.cos(B) - 1.5 * math.sin(B)

    # Local standard time meridian
    LSTM = 15 * timezone_offset

    # Time correction
    TC = 4 * (longitude - LSTM) + EoT

    # Local solar time
    LST = decimal_hour + TC/60.0

    # Hour angle
    H = 15 * (LST - 12)
    H_rad = math.radians(H)

    # Altitude angle
    sin_alpha = math.sin(lat_rad) * math.sin(decl_rad) + \
                math.cos(lat_rad) * math.cos(decl_rad) * math.cos(H_rad)
    alpha_rad = math.asin(sin_alpha)
    alpha_deg = math.degrees(alpha_rad)

    # Azimuth calculation
    cos_alpha = math.cos(alpha_rad)
    if cos_alpha == 0:
        azimuth_deg = float('nan')  # sun directly overhead
    else:
        sin_az = -math.cos(decl_rad) * math.sin(H_rad) / cos_alpha
        cos_az = (math.sin(decl_rad) - math.sin(lat_rad) * math.sin(alpha_rad)) / (math.cos(lat_rad) * cos_alpha)
        az_rad = math.atan2(sin_az, cos_az)
        azimuth_deg = math.degrees(az_rad)
        # Adjust to 0-360°
        azimuth_deg = (azimuth_deg + 360) % 360

    return alpha_deg, azimuth_deg


solar_altitude_azimuth(year = 2025, month = 7, day = 31, hour = 21, minute = 11, latitude = 55.67594, longitude = 12.56553, timezone_offset=2)


(-0.030542083520530176, 303.6307900402488)